# RedAgent: Laya-Powered Crypto Options Decision Engine
### Fast System 1 Inference & Fine-Tuning on Google Colab

This notebook configures, evaluates, and fine-tunes **Laya** (`convaiinnovations/laya`)—an open-weight, non-autoregressive System 1 decision model based on **ModernBERT-large** (~421M parameters).

### Why Laya for RedAgent?
- **Non-Autoregressive**: Zero token-by-token generation; delivers calibrated decisions in a single forward pass (~15–33ms).
- **Zero Hallucination**: Directly classifies structured state into typed decisions (`choice`, `score`, `noul`).
- **Ultra-Lightweight**: Runs in-process with <1.5 GB RAM/VRAM; trains in ~5–10 minutes on a free Colab T4 GPU.
- **Local Mac & VPS Ready**: Exportable to ONNX for sub-20ms in-process execution in your trading bot.

---

## 1. Environment Setup & Hardware Check

In [1]:
# Check GPU availability
!nvidia-smi

Tue Oct 13 16:01:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Install Laya, Transformers, and Acceleration packages
!pip install -q -U laya transformers accelerate peft datasets scipy scikit-learn
!git clone https://github.com/NandhaKishorM/laya.git || true

!pip install -q -U "pyarrow>=19,<24" laya transformers accelerate peft datasets scipy scikit-learn
print('✅ Packages installed. Verifying laya-train CLI tool:')
!laya-train --help | head -n 12

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
fatal: destination path 'laya' already exists and is not an empty directory.
✅ Packages installed. Verifying laya-train CLI tool:
usage: laya-train [-h] --data DATA [--eval EVAL_DATA] [--out OUTPUT_DIR]
                  [--base MODEL_DIR] [--text-column TEXT_COLUMN]
                  [--label-column LABEL_COLUMN] [--question-id QUESTION_ID]
                  [--instructions INSTRUCTIONS] [--epochs EPOCHS]
                  [--micro-batch MICRO_BATCH] [--grad-accum GRAD_ACCUM]
                  [--encoder-lr ENCODER_LR] [--head-lr HEAD_LR]
                  [--loss {soft-ce,rlcd}] [--label-smoothing LABEL_SMOOTHING]
                  [--shuffle-options] [--freeze-encoder]
                  [--target-error TARGET_ERROR

## 2. Google Drive Mounting & Workspace Directory

In [8]:
import os

# If running on Google Colab, mount Drive to persist data and checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/redagent'
except Exception:
    # Fallback to local directory
    DRIVE_DIR = os.path.abspath('./drive_workspace')

DATA_DIR = os.path.join(DRIVE_DIR, 'data')
MODEL_DIR = os.path.join(DRIVE_DIR, 'models/laya_options_agent')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Workspace directory: {DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace directory: /content/drive/MyDrive/redagent


## 3. Zero-Shot Verification with Base Laya (`convaiinnovations/laya`)
Test how base Laya evaluates a sample crypto call option snapshot using the high-level `Router` interface.

In [11]:
from laya import Router

# Initialize Laya router
router = Router(preload=True)

# Example synthesized market state generated by RedAgent's quant engine
sample_state = """[TIMESTAMP: 2026-10-10T17:15:00Z]
CONTRACT: BTC-24OCT26-68000-C (DTE: 3.8 days)
UNDERLYING: Spot $65,850 | Strike $68,000 | Moneyness: OTM (Delta: 0.34)
GREEKS: Delta: 0.34, Gamma: 0.000045, Theta: -$138/day, Vega: $36.2
VOLATILITY: IV: 47.8%, 30d Realized Vol: 42.1%, IV Rank: 24% (Low IV Expansion Zone)
FLOW & SENTIMENT: 8h Funding: +0.003%, 24h Call/Put Ratio: 1.52, Spot Momentum: BULLISH_BREAKOUT
ORDERBOOK & FRICTION:
- Best Bid: $715, Best Ask: $735, Spread: 2.76%
- Round-trip Taker Fee: $39.20
- 24h Theta Drag: $138.00
- Breakeven Spot Move: +1.78%
- Net Projected Return at +2.5% Spot Move: +19.2% (Well above 3% minimum hurdle)
"""

# Define typed questions according to System 1 decision contract
questions = {
    "should_buy_call": {
        "type": "noul",
        "instructions": "Based on the market state, Greeks, and fee simulation, should a trader enter this Call option to achieve >= 3% net profit after fees and slippage?"
    },
    "expected_net_return": {
        "type": "choice",
        "instructions": "What is the expected net return bracket of this Call option trade after all friction?",
        "criteria": [
            "loss_or_negative",
            "breakeven_to_2pct",
            "target_3pct_to_10pct",
            "explosive_above_10pct"
        ]
    },
    "risk_regime": {
        "type": "choice",
        "instructions": "What is the primary risk regime governing this contract?",
        "criteria": [
            "high_theta_decay",
            "iv_crush_risk",
            "favorable_low_vol_expansion",
            "trending_spot_momentum"
        ]
    }
}

# Run single-pass System 1 inference
result = router.predict(sample_state, questions)
answers = result["answers"]

print("=== Laya Zero-Shot Decision Output ===")
for q_id, resp in answers.items():
    print(f"{q_id}: {resp}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

=== Laya Zero-Shot Decision Output ===
should_buy_call: {'type': 'noul', 'noul': 0.9634, 'confidence': 0.9634, 'answer_confidence': 0.9634, 'action': {'act_probability': 1.0}}
expected_net_return: {'type': 'choice', 'choice': 'explosive_above_10pct', 'probabilities': {'loss_or_negative': 0.0482, 'breakeven_to_2pct': 0.3781, 'target_3pct_to_10pct': 0.1932, 'explosive_above_10pct': 0.3806}, 'confidence': 0.135, 'answer_confidence': 0.3806, 'action': {'act_probability': 1.0}}
risk_regime: {'type': 'choice', 'choice': 'favorable_low_vol_expansion', 'probabilities': {'high_theta_decay': 0.199, 'iv_crush_risk': 0.2095, 'favorable_low_vol_expansion': 0.3908, 'trending_spot_momentum': 0.2006}, 'confidence': 0.0347, 'answer_confidence': 0.3908, 'action': {'act_probability': 1.0}}


## 4. Dataset Loading & Validation
Verify the historical training set generated by RedAgent's historical simulation pipeline.

In [9]:
import json

train_file = os.path.join(DATA_DIR, 'train.jsonl')
val_file = os.path.join(DATA_DIR, 'val.jsonl')

# Check if dataset exists or generate a small synthetic verification sample
if not os.path.exists(train_file):
    print("Historical train.jsonl not found. Generating verification sample...")
    sample_data = [
        {
            "state": sample_state,
            "questions": questions,
            "labels": {
                "should_buy_call": "yes",
                "expected_net_return": "target_3pct_to_10pct",
                "risk_regime": "trending_spot_momentum"
            }
        }
    ]
    with open(train_file, 'w') as f:
        for row in sample_data:
            f.write(json.dumps(row) + '\n')
    with open(val_file, 'w') as f:
        for row in sample_data:
            f.write(json.dumps(row) + '\n')
    print(f"Sample dataset created at {train_file}")
else:
    with open(train_file, 'r') as f:
        count = sum(1 for _ in f)
    print(f"Verified training dataset: {count} samples.")

Verified training dataset: 3330 samples.


## 5. Fine-Tuning Laya on Crypto Options Data
Because Laya is an encoder (~421M parameters), fine-tuning on a Colab T4 GPU trains extremely fast (~5–10 minutes) with minimal memory overhead. -failed due to out of memory error

In [ ]:
import torch
from laya.train import TrainConfig, finetune

MODEL_NAME = 'convaiinnovations/laya'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Starting Laya fine-tuning on: {device.upper()}')
print(f'Training data:   {train_file}')
print(f'Validation data: {val_file}')
print(f'Model output:    {MODEL_DIR}')

config = TrainConfig(
    epochs=4,
    micro_batch=4,
    grad_accum=4,
    seed=42,
    eval_data=val_file,
)

summary = finetune(
    data=train_file,
    model_dir=MODEL_NAME,
    output_dir=MODEL_DIR,
    config=config,
    device=device,
)
print('Fine-tuning summary:', summary)


Starting Laya fine-tuning on: CUDA
Training data:   /content/drive/MyDrive/redagent/data/train.jsonl
Validation data: /content/drive/MyDrive/redagent/data/val.jsonl
Model output:    /content/drive/MyDrive/redagent/models/laya_options_agent


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

eval items 833 from /content/drive/MyDrive/redagent/data/val.jsonl (skipped {'invalid_question': 1666}, 0 overlapping training/calibration items)
train items 2997, calibration items 333, eval items 833 (/content/drive/MyDrive/redagent/data/val.jsonl), skipped {'invalid_question': 6660}, device cuda
optimizer budget: 2997 train item(s), effective batch up to 16 (micro_batch 4 x grad_accum 4), 752 optimizer updates over 4 epoch(s)


In [15]:
import gc
import torch
import os

# 1. Clear GPU Memory from earlier steps
try:
    del router
    del result
    del answers
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# 2. Configure PyTorch to combat fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Re-run fine-tuning with safer, lower micro-batch size
from laya.train import TrainConfig, finetune

MODEL_NAME = 'convaiinnovations/laya'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Starting Laya fine-tuning on: {device.upper()}')

# We reduce micro_batch from 4 to 2, and increase grad_accum to 8
# This maintains the effective batch size of 16 (2 * 8 = 16) while saving half the VRAM!
config = TrainConfig(
    epochs=4,
    micro_batch=2,
    grad_accum=8,
    seed=42,
    eval_data=val_file,
)

summary = finetune(
    data=train_file,
    model_dir=MODEL_NAME,
    output_dir=MODEL_DIR,
    config=config,
    device=device,
)
print('Fine-tuning summary:', summary)

Starting Laya fine-tuning on: CUDA


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

eval items 833 from /content/drive/MyDrive/redagent/data/val.jsonl (skipped {'invalid_question': 1666}, 0 overlapping training/calibration items)
train items 2997, calibration items 333, eval items 833 (/content/drive/MyDrive/redagent/data/val.jsonl), skipped {'invalid_question': 6660}, device cuda
optimizer budget: 2997 train item(s), effective batch up to 16 (micro_batch 2 x grad_accum 8), 752 optimizer updates over 4 epoch(s)


/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 13.66 GiB is allocated by PyTorch, and 768.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# We use a subprocess call to train in an isolated memory environment.
# This bypasses all active variables, models, and leaks currently held by this notebook's kernel.
!laya-train \
    --data "{train_file}" \
    --eval "{val_file}" \
    --base "{MODEL_NAME}" \
    --out "{MODEL_DIR}" \
    --epochs 4 \
    --micro-batch 2 \
    --grad-accum 8 \
    --seed 42

## 6. Model Evaluation & Calibration Verification
Verify that output probabilities are well-calibrated (Brier score and ROC-AUC) so the trading bot's threshold (e.g. $P(\text{win}) \ge 0.75$) reflects actual historical win rates.

In [ ]:
from laya import load

# Test loaded checkpoint
agent = load(MODEL_DIR if os.path.exists(os.path.join(MODEL_DIR, 'config.json')) else "convaiinnovations/laya")
test_res = agent.predict(sample_state, questions)

print("=== Post-Training Evaluation ===")
print(json.dumps(test_res, indent=2))

# Check confidence
buy_decision = test_res["answers"]["should_buy_call"]
confidence = buy_decision.get("confidence", 0.0)
print(f"\nCall Buy Recommendation: {buy_decision.get('choice')} with confidence: {confidence:.2%}")
if confidence >= 0.75 and buy_decision.get('choice') == 'yes':
    print("Signal Status: VALIDATED -> Trigger Telegram/Discord Alert & Paper Execution!")
else:
    print("Signal Status: FILTERED OUT -> Does not meet conviction hurdle.")

## 7. Next Steps for RedAgent
1. Download the fine-tuned checkpoint folder from `/content/drive/MyDrive/redagent/models/laya_options_agent` to your local Mac directory (`redagent/models/laya_options_agent`).
2. The live scanner loop (`src/signals/signal_engine.py`) loads this checkpoint via `from laya import load` for in-process, sub-20ms decisions without spinning up a heavy GPU server.